In [1]:
"""
Hotel Room Booking System
-------------------------
GUI: Tkinter (ships with Python)  |  Database: SQLite (ships with Python)
No extra packages needed.  Run with:  python hotel_booking_system.py
"""

import sqlite3
import tkinter as tk
from datetime import date, datetime
from tkinter import messagebox, ttk

DB_FILE = "hotel_system.db"
DATE_FMT = "%Y-%m-%d"
ROOM_TYPES = ["Single", "Double", "Deluxe", "Suite"]


# ----------------------------------------------------------------------
# Database layer
# ----------------------------------------------------------------------
class Database:
    def __init__(self, path=DB_FILE):
        self.conn = sqlite3.connect(path)
        self.conn.execute("PRAGMA foreign_keys = ON")
        self._create_tables()
        self._seed_rooms()

    def _create_tables(self):
        self.conn.executescript(
            """
            CREATE TABLE IF NOT EXISTS rooms (
                id     INTEGER PRIMARY KEY AUTOINCREMENT,
                number TEXT UNIQUE NOT NULL,
                type   TEXT NOT NULL,
                price  REAL NOT NULL
            );
            CREATE TABLE IF NOT EXISTS bookings (
                id        INTEGER PRIMARY KEY AUTOINCREMENT,
                room_id   INTEGER NOT NULL,
                guest     TEXT NOT NULL,
                phone     TEXT NOT NULL,
                check_in  TEXT NOT NULL,
                check_out TEXT NOT NULL,
                total     REAL NOT NULL,
                created   TEXT NOT NULL,
                FOREIGN KEY (room_id) REFERENCES rooms(id) ON DELETE CASCADE
            );
            """
        )
        self.conn.commit()

    def _seed_rooms(self):
        if self.conn.execute("SELECT COUNT(*) FROM rooms").fetchone()[0] == 0:
            sample = [
                ("101", "Single", 1500), ("102", "Single", 1500),
                ("201", "Double", 2500), ("202", "Double", 2500),
                ("301", "Deluxe", 3800), ("302", "Deluxe", 3800),
                ("401", "Suite", 6000),
            ]
            self.conn.executemany(
                "INSERT INTO rooms (number, type, price) VALUES (?, ?, ?)", sample
            )
            self.conn.commit()

    # ---- rooms ----
    def add_room(self, number, rtype, price):
        self.conn.execute(
            "INSERT INTO rooms (number, type, price) VALUES (?, ?, ?)",
            (number, rtype, price),
        )
        self.conn.commit()

    def delete_room(self, room_id):
        today = date.today().strftime(DATE_FMT)
        active = self.conn.execute(
            "SELECT COUNT(*) FROM bookings WHERE room_id=? AND check_out > ?",
            (room_id, today),
        ).fetchone()[0]
        if active:
            raise ValueError("This room has current or upcoming bookings.")
        self.conn.execute("DELETE FROM rooms WHERE id=?", (room_id,))
        self.conn.commit()

    def all_rooms_with_status(self):
        today = date.today().strftime(DATE_FMT)
        return self.conn.execute(
            """
            SELECT r.id, r.number, r.type, r.price,
                   CASE WHEN EXISTS (
                        SELECT 1 FROM bookings b
                        WHERE b.room_id = r.id
                          AND b.check_in <= ? AND b.check_out > ?
                   ) THEN 'Occupied' ELSE 'Available' END
            FROM rooms r ORDER BY r.number
            """,
            (today, today),
        ).fetchall()

    def available_rooms(self, check_in, check_out, rtype=None):
        """Rooms with no booking overlapping [check_in, check_out)."""
        query = """
            SELECT id, number, type, price FROM rooms r
            WHERE NOT EXISTS (
                SELECT 1 FROM bookings b
                WHERE b.room_id = r.id
                  AND b.check_in < ? AND b.check_out > ?
            )
        """
        params = [check_out, check_in]
        if rtype and rtype != "Any":
            query += " AND r.type = ?"
            params.append(rtype)
        query += " ORDER BY r.number"
        return self.conn.execute(query, params).fetchall()

    # ---- bookings ----
    def is_room_free(self, room_id, check_in, check_out):
        clash = self.conn.execute(
            """SELECT COUNT(*) FROM bookings
               WHERE room_id=? AND check_in < ? AND check_out > ?""",
            (room_id, check_out, check_in),
        ).fetchone()[0]
        return clash == 0

    def add_booking(self, room_id, guest, phone, check_in, check_out, total):
        self.conn.execute(
            """INSERT INTO bookings
               (room_id, guest, phone, check_in, check_out, total, created)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (room_id, guest, phone, check_in, check_out, total,
             datetime.now().strftime("%Y-%m-%d %H:%M")),
        )
        self.conn.commit()

    def all_bookings(self, search=""):
        like = f"%{search}%"
        return self.conn.execute(
            """
            SELECT b.id, r.number, r.type, b.guest, b.phone,
                   b.check_in, b.check_out, b.total
            FROM bookings b JOIN rooms r ON r.id = b.room_id
            WHERE b.guest LIKE ? OR r.number LIKE ? OR b.phone LIKE ?
            ORDER BY b.check_in DESC
            """,
            (like, like, like),
        ).fetchall()

    def cancel_booking(self, booking_id):
        self.conn.execute("DELETE FROM bookings WHERE id=?", (booking_id,))
        self.conn.commit()


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def parse_dates(check_in_text, check_out_text):
    try:
        cin = datetime.strptime(check_in_text.strip(), DATE_FMT).date()
        cout = datetime.strptime(check_out_text.strip(), DATE_FMT).date()
    except ValueError:
        raise ValueError("Dates must be in YYYY-MM-DD format (e.g. 2026-12-25).")
    if cin < date.today():
        raise ValueError("Check-in date cannot be in the past.")
    if cout <= cin:
        raise ValueError("Check-out must be after check-in.")
    return cin, cout


def make_tree(parent, columns, widths, height=12):
    frame = ttk.Frame(parent)
    tree = ttk.Treeview(frame, columns=columns, show="headings", height=height)
    for col, w in zip(columns, widths):
        tree.heading(col, text=col)
        tree.column(col, width=w, anchor="center")
    scroll = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    tree.configure(yscrollcommand=scroll.set)
    tree.pack(side="left", fill="both", expand=True)
    scroll.pack(side="right", fill="y")
    return frame, tree


# ----------------------------------------------------------------------
# GUI
# ----------------------------------------------------------------------
class HotelApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Hotel Room Booking System")
        self.geometry("860x580")
        self.minsize(780, 520)
        self.db = Database()

        ttk.Style(self).theme_use("clam")

        notebook = ttk.Notebook(self)
        notebook.pack(fill="both", expand=True, padx=10, pady=10)

        self.book_tab = ttk.Frame(notebook, padding=12)
        self.rooms_tab = ttk.Frame(notebook, padding=12)
        self.bookings_tab = ttk.Frame(notebook, padding=12)
        notebook.add(self.book_tab, text="  Book a Room  ")
        notebook.add(self.rooms_tab, text="  Rooms  ")
        notebook.add(self.bookings_tab, text="  Bookings  ")
        notebook.bind("<<NotebookTabChanged>>", lambda e: self.refresh_all())

        self._build_book_tab()
        self._build_rooms_tab()
        self._build_bookings_tab()
        self.refresh_all()

    # ---------------- Book tab ----------------
    def _build_book_tab(self):
        f = self.book_tab
        form = ttk.LabelFrame(f, text="Guest & stay details", padding=10)
        form.pack(fill="x")

        self.guest_var = tk.StringVar()
        self.phone_var = tk.StringVar()
        self.cin_var = tk.StringVar(value=date.today().strftime(DATE_FMT))
        self.cout_var = tk.StringVar()
        self.type_var = tk.StringVar(value="Any")

        fields = [
            ("Guest name", self.guest_var, 0, 0),
            ("Phone", self.phone_var, 0, 2),
            ("Check-in (YYYY-MM-DD)", self.cin_var, 1, 0),
            ("Check-out (YYYY-MM-DD)", self.cout_var, 1, 2),
        ]
        for label, var, r, c in fields:
            ttk.Label(form, text=label).grid(row=r, column=c, sticky="w", padx=5, pady=5)
            ttk.Entry(form, textvariable=var, width=24).grid(row=r, column=c + 1, padx=5, pady=5)

        ttk.Label(form, text="Room type").grid(row=2, column=0, sticky="w", padx=5, pady=5)
        ttk.Combobox(
            form, textvariable=self.type_var, values=["Any"] + ROOM_TYPES,
            state="readonly", width=21,
        ).grid(row=2, column=1, padx=5, pady=5)
        ttk.Button(form, text="Search available rooms", command=self.search_rooms).grid(
            row=2, column=3, sticky="e", padx=5, pady=5
        )

        ttk.Label(f, text="Available rooms (select one):").pack(anchor="w", pady=(12, 2))
        frame, self.avail_tree = make_tree(
            f, ("ID", "Room", "Type", "Price / night"), (50, 120, 140, 140), height=8
        )
        frame.pack(fill="both", expand=True)

        bottom = ttk.Frame(f)
        bottom.pack(fill="x", pady=(10, 0))
        self.total_label = ttk.Label(bottom, text="Total: -", font=("Segoe UI", 11, "bold"))
        self.total_label.pack(side="left")
        ttk.Button(bottom, text="Book selected room", command=self.book_room).pack(side="right")
        self.avail_tree.bind("<<TreeviewSelect>>", lambda e: self.update_total())

    def search_rooms(self):
        try:
            cin, cout = parse_dates(self.cin_var.get(), self.cout_var.get())
        except ValueError as e:
            messagebox.showerror("Invalid dates", str(e))
            return
        self.avail_tree.delete(*self.avail_tree.get_children())
        rooms = self.db.available_rooms(str(cin), str(cout), self.type_var.get())
        for rid, number, rtype, price in rooms:
            self.avail_tree.insert("", "end", values=(rid, number, rtype, f"{price:.2f}"))
        if not rooms:
            messagebox.showinfo("No rooms", "No rooms are available for those dates.")
        self.total_label.config(text="Total: -")

    def update_total(self):
        sel = self.avail_tree.selection()
        if not sel:
            return
        try:
            cin, cout = parse_dates(self.cin_var.get(), self.cout_var.get())
        except ValueError:
            return
        price = float(self.avail_tree.item(sel[0])["values"][3])
        nights = (cout - cin).days
        self.total_label.config(text=f"Total: {nights} night(s) x {price:.2f} = {nights * price:.2f}")

    def book_room(self):
        guest, phone = self.guest_var.get().strip(), self.phone_var.get().strip()
        if not guest or not phone:
            messagebox.showerror("Missing details", "Please enter guest name and phone.")
            return
        if not phone.replace("+", "").replace("-", "").replace(" ", "").isdigit():
            messagebox.showerror("Invalid phone", "Phone may contain only digits, +, - and spaces.")
            return
        try:
            cin, cout = parse_dates(self.cin_var.get(), self.cout_var.get())
        except ValueError as e:
            messagebox.showerror("Invalid dates", str(e))
            return
        sel = self.avail_tree.selection()
        if not sel:
            messagebox.showerror("No room", "Search and select a room first.")
            return

        room_id, number, _, price = self.avail_tree.item(sel[0])["values"]
        # Re-check right before saving, in case something changed
        if not self.db.is_room_free(room_id, str(cin), str(cout)):
            messagebox.showerror("Unavailable", "That room was just booked. Search again.")
            self.search_rooms()
            return

        nights = (cout - cin).days
        total = nights * float(price)
        if messagebox.askyesno(
            "Confirm booking",
            f"Guest: {guest}\nRoom: {number}\n{cin} to {cout} ({nights} night(s))\n"
            f"Total: {total:.2f}\n\nConfirm booking?",
        ):
            self.db.add_booking(room_id, guest, phone, str(cin), str(cout), total)
            messagebox.showinfo("Booked", f"Room {number} booked for {guest}.")
            self.guest_var.set("")
            self.phone_var.set("")
            self.search_rooms()
            self.refresh_all()

    # ---------------- Rooms tab ----------------
    def _build_rooms_tab(self):
        f = self.rooms_tab
        frame, self.rooms_tree = make_tree(
            f, ("ID", "Room", "Type", "Price / night", "Status today"),
            (50, 110, 130, 130, 130), height=12,
        )
        frame.pack(fill="both", expand=True)

        add = ttk.LabelFrame(f, text="Add a new room", padding=10)
        add.pack(fill="x", pady=(10, 0))
        self.new_number = tk.StringVar()
        self.new_type = tk.StringVar(value=ROOM_TYPES[0])
        self.new_price = tk.StringVar()

        ttk.Label(add, text="Room no.").grid(row=0, column=0, padx=5)
        ttk.Entry(add, textvariable=self.new_number, width=10).grid(row=0, column=1, padx=5)
        ttk.Label(add, text="Type").grid(row=0, column=2, padx=5)
        ttk.Combobox(add, textvariable=self.new_type, values=ROOM_TYPES,
                     state="readonly", width=10).grid(row=0, column=3, padx=5)
        ttk.Label(add, text="Price / night").grid(row=0, column=4, padx=5)
        ttk.Entry(add, textvariable=self.new_price, width=10).grid(row=0, column=5, padx=5)
        ttk.Button(add, text="Add room", command=self.add_room).grid(row=0, column=6, padx=8)
        ttk.Button(add, text="Delete selected", command=self.delete_room).grid(row=0, column=7, padx=5)

    def add_room(self):
        number = self.new_number.get().strip()
        try:
            price = float(self.new_price.get())
            if price <= 0:
                raise ValueError
        except ValueError:
            messagebox.showerror("Invalid price", "Enter a positive number for the price.")
            return
        if not number:
            messagebox.showerror("Missing room number", "Enter a room number.")
            return
        try:
            self.db.add_room(number, self.new_type.get(), price)
        except sqlite3.IntegrityError:
            messagebox.showerror("Duplicate", f"Room {number} already exists.")
            return
        self.new_number.set("")
        self.new_price.set("")
        self.refresh_all()

    def delete_room(self):
        sel = self.rooms_tree.selection()
        if not sel:
            messagebox.showinfo("Select a room", "Select a room to delete.")
            return
        room_id, number = self.rooms_tree.item(sel[0])["values"][:2]
        if messagebox.askyesno("Delete room", f"Delete room {number}?"):
            try:
                self.db.delete_room(room_id)
            except ValueError as e:
                messagebox.showerror("Cannot delete", str(e))
                return
            self.refresh_all()

    # ---------------- Bookings tab ----------------
    def _build_bookings_tab(self):
        f = self.bookings_tab
        top = ttk.Frame(f)
        top.pack(fill="x", pady=(0, 8))
        ttk.Label(top, text="Search (guest / room / phone):").pack(side="left")
        self.search_var = tk.StringVar()
        entry = ttk.Entry(top, textvariable=self.search_var, width=28)
        entry.pack(side="left", padx=6)
        entry.bind("<KeyRelease>", lambda e: self.load_bookings())

        frame, self.bookings_tree = make_tree(
            f, ("ID", "Room", "Type", "Guest", "Phone", "Check-in", "Check-out", "Total"),
            (40, 60, 80, 140, 110, 95, 95, 90), height=14,
        )
        frame.pack(fill="both", expand=True)
        ttk.Button(f, text="Cancel selected booking", command=self.cancel_booking).pack(
            anchor="e", pady=(10, 0)
        )

    def load_bookings(self):
        self.bookings_tree.delete(*self.bookings_tree.get_children())
        for row in self.db.all_bookings(self.search_var.get().strip()):
            row = list(row)
            row[-1] = f"{row[-1]:.2f}"
            self.bookings_tree.insert("", "end", values=row)

    def cancel_booking(self):
        sel = self.bookings_tree.selection()
        if not sel:
            messagebox.showinfo("Select a booking", "Select a booking to cancel.")
            return
        values = self.bookings_tree.item(sel[0])["values"]
        if messagebox.askyesno("Cancel booking", f"Cancel booking #{values[0]} for {values[3]}?"):
            self.db.cancel_booking(values[0])
            self.refresh_all()

    # ---------------- Refresh ----------------
    def refresh_all(self):
        self.rooms_tree.delete(*self.rooms_tree.get_children())
        for rid, number, rtype, price, status in self.db.all_rooms_with_status():
            self.rooms_tree.insert("", "end", values=(rid, number, rtype, f"{price:.2f}", status))
        self.load_bookings()


if __name__ == "__main__":
    HotelApp().mainloop()

TclError: no display name and no $DISPLAY environment variable